In [7]:
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

# --- CONFIGURATION ---
# These parameters MUST match the ones used in the training/generation scripts.
SEQ_LEN = 12      # Input history length
PRED_LEN = 12     # Output forecast horizon
DATA_PATH = '../../../ICL4DT/data/time_series_datasets/ETTm2.csv'
HTI_DATA_DIR = 'hti_data_long'

# The quantiles of the expert models you want to visualize
QUANTILES_TO_LOAD = [0.01, 0.1, 0.25, 0.5, 0.75,0.9, 0.99]

print("Loading original dataset...")
df = pd.read_csv(DATA_PATH)
data = df['OT'].values.astype(float)

print(f"Full dataset shape: {data.shape}")

# Recreate the exact train/val/test split to fit the scaler correctly
train_split_idx = int(len(data) * 0.7)
val_split_idx = int(len(data) * 0.98)

# Isolate the original, unscaled test data for ground truth comparison
original_test_data = data[val_split_idx:]

# Fit the scaler ONLY on the training data to prevent data leakage
print("Fitting MinMaxScaler on the training data portion...")
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(data[:train_split_idx].reshape(-1, 1))

hti_datasets = {}
all_forecasts_unscaled = {}

print("Loading HTI datasets and unscaling forecasts...")

for q in QUANTILES_TO_LOAD:
    # Construct filename (e.g., hti_data_q05.pt)
    filename = f"hti_data_q{str(q).replace('.', '')}.pt"
    
    try:
        # Load the entire [history, forecast] tensor
        hti_datasets[q] = torch.load(filename)
        data = hti_datasets[q]
        print(f" -> Loaded '{filename}' with shape: {hti_datasets[q].shape}")
        data_unscaled = scaler.inverse_transform(data)
        data_unscaled = torch.tensor(data_unscaled, dtype=torch.float32)
        all_forecasts_unscaled[q] = data_unscaled
        
    except FileNotFoundError:
        print(f" -> WARNING: Could not find file {filename}. Skipping.")

print("\nForecasts are now unscaled and ready for plotting.")


Loading original dataset...
Full dataset shape: (69680,)
Fitting MinMaxScaler on the training data portion...
Loading HTI datasets and unscaling forecasts...
 -> Loaded 'hti_data_q001.pt' with shape: torch.Size([1371, 24])
 -> Loaded 'hti_data_q01.pt' with shape: torch.Size([1371, 24])
 -> Loaded 'hti_data_q025.pt' with shape: torch.Size([1371, 24])
 -> Loaded 'hti_data_q05.pt' with shape: torch.Size([1371, 24])
 -> Loaded 'hti_data_q075.pt' with shape: torch.Size([1371, 24])
 -> Loaded 'hti_data_q09.pt' with shape: torch.Size([1371, 24])
 -> Loaded 'hti_data_q099.pt' with shape: torch.Size([1371, 24])

Forecasts are now unscaled and ready for plotting.


In [8]:
all_forecasts_unscaled

{0.01: tensor([[34.1700, 34.3895, 35.0485,  ..., 36.5375, 35.9743, 35.6838],
         [34.3895, 35.0485, 35.4885,  ..., 37.8795, 37.2787, 36.9889],
         [35.0485, 35.4885, 36.1475,  ..., 38.5891, 37.9721, 37.6836],
         ...,
         [47.0850, 47.0850, 47.0850,  ..., 40.2748, 39.7877, 39.5586],
         [47.0850, 47.0850, 47.0850,  ..., 40.0915, 39.6053, 39.3746],
         [47.0850, 47.0850, 47.0850,  ..., 39.9170, 39.4261, 39.1914]]),
 0.1: tensor([[34.1700, 34.3895, 35.0485,  ..., 42.1801, 41.9807, 41.6140],
         [34.3895, 35.0485, 35.4885,  ..., 43.5319, 43.3324, 42.9721],
         [35.0485, 35.4885, 36.1475,  ..., 43.9859, 43.7739, 43.3791],
         ...,
         [47.0850, 47.0850, 47.0850,  ..., 45.3385, 45.1303, 44.9083],
         [47.0850, 47.0850, 47.0850,  ..., 45.2381, 45.0198, 44.7918],
         [47.0850, 47.0850, 47.0850,  ..., 45.1977, 44.9727, 44.7400]]),
 0.25: tensor([[34.1700, 34.3895, 35.0485,  ..., 42.7842, 42.5934, 42.4694],
         [34.3895, 35.0485, 

In [9]:
combined = torch.stack([all_forecasts_unscaled[q] for q in QUANTILES_TO_LOAD], dim=0)

torch.save(combined, 'hti_data_combined.pt')

In [10]:
combined.shape

torch.Size([7, 1371, 24])

In [11]:
combined

tensor([[[34.1700, 34.3895, 35.0485,  ..., 36.5375, 35.9743, 35.6838],
         [34.3895, 35.0485, 35.4885,  ..., 37.8795, 37.2787, 36.9889],
         [35.0485, 35.4885, 36.1475,  ..., 38.5891, 37.9721, 37.6836],
         ...,
         [47.0850, 47.0850, 47.0850,  ..., 40.2748, 39.7877, 39.5586],
         [47.0850, 47.0850, 47.0850,  ..., 40.0915, 39.6053, 39.3746],
         [47.0850, 47.0850, 47.0850,  ..., 39.9170, 39.4261, 39.1914]],

        [[34.1700, 34.3895, 35.0485,  ..., 42.1801, 41.9807, 41.6140],
         [34.3895, 35.0485, 35.4885,  ..., 43.5319, 43.3324, 42.9721],
         [35.0485, 35.4885, 36.1475,  ..., 43.9859, 43.7739, 43.3791],
         ...,
         [47.0850, 47.0850, 47.0850,  ..., 45.3385, 45.1303, 44.9083],
         [47.0850, 47.0850, 47.0850,  ..., 45.2381, 45.0198, 44.7918],
         [47.0850, 47.0850, 47.0850,  ..., 45.1977, 44.9727, 44.7400]],

        [[34.1700, 34.3895, 35.0485,  ..., 42.7842, 42.5934, 42.4694],
         [34.3895, 35.0485, 35.4885,  ..., 43

In [12]:
combined_min = combined.min().item()
combined_max = combined.max().item()
print(f"Min value in combined: {combined_min}")
print(f"Max value in combined: {combined_max}")

Min value in combined: 20.033174514770508
Max value in combined: 57.20264434814453
